# 09. Juntando tudo

Para fechar a trilha, um experimento de ponta a ponta: **planejar** o tamanho da amostra (`power_analysis`), **rodar** com diagnóstico automático (`ExperimentPipeline`) e **relatar** (`ExperimentReport`).

## 1. Planejar: quantas unidades?

`power_analysis` resolve um de (n, MDE, power) dados os outros dois. Aqui pedimos o `n` para detectar um efeito de `0.3` com 80% de poder.

In [ ]:
from skxperiments.design.power import power_analysis

plan = power_analysis(mde=0.3, power=0.8, std=1.0, alpha=0.05)
print(f"n necessario: {plan.n_total} (cerca de {plan.n_treated} por grupo)")

## 2. Rodar: pipeline com diagnóstico automático

O `ExperimentPipeline` roda os diagnósticos (SRM por padrão) e a inferência num passo só, e empacota tudo num `PipelineResult`.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.diagnostics import SRMTest
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI
from skxperiments.pipeline import ExperimentPipeline

n = plan.n_total
rng = np.random.default_rng(0)
y0 = rng.normal(size=n)
tau = 0.3
y1 = y0 + tau
df = pd.DataFrame({"x": rng.normal(size=n)})
design = CRD(p=0.5, seed=0)
a = design.randomize(df)
t = a.data_[a.treatment_col_].to_numpy()
data = a.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
a = CRDAssignment(data=data, treatment_col=a.treatment_col_, design=design, seed=0)

pr = ExperimentPipeline(
    inference=NeymanCI(estimator=DifferenceInMeans(outcome_col="y")),
    diagnostics=[SRMTest()],
).run(a)
print(f"ATE={pr.results.ate:.3f}, p={pr.results.p_value:.4f}, flag={pr.flagged}")

## 3. Relatar

`ExperimentReport` gera um HTML autocontido com a tabela de resultados, os diagnósticos e os gráficos embutidos. Aqui renderizamos inline; em produção, use `report.save("relatorio.html")`.

In [ ]:
from IPython.display import HTML

from skxperiments.reporting import ExperimentReport

report = ExperimentReport(pr, title="Relatorio do experimento")
HTML(report.to_html())

## Fim da trilha

Você percorreu desenho, estimação, três formas de inferência, redução de variância, equilíbrio, blocagem, fatorial, múltiplos testes, diagnósticos e relatório. É o fluxo completo da biblioteca.

Esta é a trilha **for_starters** (dados simulados, didática). Em breve, uma trilha com **dados reais** mostra a lib em casos do mundo real.